# Dashboard Contoso

## Importeren libraries en datasets

In [2]:
#Importeren van libraries
import matplotlib.pyplot as plt
import pandas as pd
import squarify
import seaborn as sns
import plotly.graph_objs as go
import plotly.io as pio
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

In [3]:
#Importeren parquet files uit ETL.
sales_df = pd.read_parquet(r"PQ/Contoso_sales.parquet")
product_df = pd.read_parquet(r"PQ/Contoso_product.parquet")
calendar_df = pd.read_parquet(r"PQ/Contoso_calendar.parquet")
geography_df = pd.read_parquet(r"PQ/Contoso_geography.parquet")
stores_df = pd.read_parquet(r"PQ/Contoso_stores.parquet")
promotions_df = pd.read_parquet(r"PQ/Contoso_promotions.parquet")
productcat_df = pd.read_parquet(r"PQ/Contoso_productcat.parquet")
channel_df = pd.read_parquet(r"PQ/Contoso_channel.parquet")
GDP_df = pd.read_parquet(r"PQ/Contose_GDP.parquet")

## Sales Dashboard

### Measures

In [4]:
#In dit onderdeel worden de measures uit PowerBI nagemaakt. 
#De waarden voor revenue en cost worden eerst totaal berekend. Vervolgens worden Gross Margin en Gross margin in procenten op basis hiervan berekend.

revenue = sales_df["SalesAmountRet"].sum()
#Sales berekening
total_cost = sales_df["TotalCostRet"].sum()
#Cost berekening
gross_margin = revenue - total_cost
#Gross margin berekening
gross_margin_pct = (gross_margin / revenue)*100
#Gross margin in percentage

### KPI's

In [5]:
#Revenue KPI
fig = go.Figure(
    #Maakt de figuur
    go.Indicator(
        #Maakt de KPI-visual
        mode="number",
        value=revenue,
        title={"text": "Revenue"},
        number={
            "prefix": "€",
            #Zet euro-teken voor het getal
            "valueformat": ",.0f"
            #Rond af op 0 decimalen.
        }
    )
)
fig.update_layout(
    height=200,
    width=300
)
#Formaat instellen
fig.show()
#Cost KPI
fig = go.Figure(
    #Maakt de figuur
    go.Indicator(
        #Maakt de KPI-visual
        mode="number",
        value=total_cost,
        title={"text": "Cost"},
        number={
            "prefix": "€",
            #Zet euro-teken voor het getal
            "valueformat": ",.0f"
            #Rond af op hele getallen en gebruikt komma's als scheidingstekens tussen duizendtallen.
        }
    )
)
fig.update_layout(
    height=200,
    width=300
)
#Formaat instellen
fig.show()
#Gross margin KPI
fig = go.Figure(
    #Maakt de figuur
    go.Indicator(
        #Maakt de KPI-visual
        mode="number",
        value=gross_margin,
        title={"text": "Gross Margin"},
        number={
            "prefix": "€",
            #Zet euro-teken voor het getal
            "valueformat": ",.0f"
            #Rond af op hele getallen en gebruikt komma's als scheidingstekens tussen duizendtallen.
        }
    )
)
fig.update_layout(
    height=200,
    width=300
)
#Formaat instellen
fig.show()
#Gross margin percentage KPI
fig = go.Figure(
    #Maakt de figuur
    go.Indicator(
        #Maakt de KPI-visual
        mode="number",
        value=gross_margin_pct,
        title={"text": "Gross Margin percentage"},
        number={
            "suffix": "%",
            # %-teken na het percentage.
            "valueformat": ".01f"
            #Rond het percentage af op 1 decimaal.

        }
    )
)
fig.update_layout(
    height=200,
    width=300
)
#Formaat instellen

fig.show()


### Revenue over time

In [6]:
monthly_revenue = sales_df.resample("ME", on="Date").agg(Revenue=("SalesAmountRet","sum")).reset_index()
#Totale omzet aggregeren per maand en reset de index, zodat datum niet als index gebruikt blijft worden
monthly_revenue
monthly_revenue["Year"] = monthly_revenue["Date"].dt.year.astype(str)
#Maakt een kolom met de dataframe in jaar.
monthly_revenue["Month"] = monthly_revenue["Date"].dt.month
#Maakt een kolom met de maand van het jaar in nummers.

In [7]:
fig = px.line(
    monthly_revenue,
    x="Month",
    y="Revenue",
    color="Year",
    title="Revenue over time"
)
#Figuur maken met maanden op de x-as, omzet op de y-as en aparte lijnen voor ieder jaar.

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Revenue",
    height=400,
    width=1000
)
#Omschrijvingen van assen toevoegen.

fig.update_yaxes(
    tickprefix="€",
    tickformat=",.0f"
)
#Valuta maken van waarden op de y-as

fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(1, 13)),
    ticktext=[
        "Jan", "Feb", "Mar", "Apr",
        "May", "Jun", "Jul", "Aug",
        "Sep", "Oct", "Nov", "Dec"
    ]
)
#Nummers omzetten in afkortingen van maanden.

fig.show()

### Revenue by channel

In [8]:
channel_sales = sales_df.merge(right=channel_df, on="ChannelKey", how="left", indicator=True)
#Aparte Dataframe maken waarbij Channel wordt gemerged met de salesdataset.
print(channel_sales.shape)
print(sales_df.shape)
#Shapes printen om te controleren of er evenveel records in de set zitten.

(2282482, 22)
(2282482, 20)


In [9]:
channel_revenue = (
    channel_sales
    .groupby("ChannelName", as_index=False)
    .agg(Revenue=("SalesAmountRet", "sum"))
    .sort_values("Revenue", ascending=True)
)

In [10]:
fig = px.bar(
    #Bar chart maken met behulp van channel_revenue dataframe.
    channel_revenue,
    x="Revenue",
    y="ChannelName",
    orientation="h",
    title="Revenue by sales channel"
)

fig.update_layout(
    xaxis_title="Revenue",
    yaxis_title="Sales channel",
    height=400,
    width = 800
)
#Titels voor axes instellen.
#Hoogte en wijdte van de chart instellen.

fig.update_xaxes(
    tickprefix="€",
    tickformat=",.0f"
)

fig.show()

### Revenue by product category

In [11]:
product_sales = (
    sales_df
    .merge(
        product_df,
        on="ProductKey",
        how="left",
        indicator=True
    )
    .merge(
        productcat_df,
        on="ProductSubcategoryKey", 
        how="left",
    )
)
print(f"product sales shape is {product_sales.shape}")
print(f"sales shape is {sales_df.shape}")

product sales shape is (2282482, 34)
sales shape is (2282482, 20)


In [12]:
category_revenue = (
    product_sales
    .groupby("ProductCategory", as_index=False)
    .agg(Revenue=("SalesAmountRet", "sum"))
    .sort_values("Revenue", ascending=True)
)

In [13]:
fig = px.pie(
    category_revenue,
    values="Revenue",
    names="ProductCategory",
    hole=0.8,
    title="Revenue by product category"
)

fig.update_layout(
    height=450,
    width=800
)

fig.update_traces(
    textposition="outside",
    textinfo="percent+label",
    showlegend=False
    
)


fig.show()

### GDP per capita versus revenue

In [14]:
sales_geo = sales_df.merge(
    geography_df,
    on="GeographyKey",
    how="left"
)

sales_geo["Year"] = sales_geo["Date"].dt.year
sales_geo.info()

<class 'pandas.DataFrame'>
RangeIndex: 2282482 entries, 0 to 2282481
Data columns (total 24 columns):
 #   Column             Dtype         
---  ------             -----         
 0   SalesKey           int64         
 1   Date               datetime64[us]
 2   ChannelKey         int64         
 3   StoreKey           int64         
 4   ProductKey         int64         
 5   PromotionKey       int64         
 6   UnitCost           float64       
 7   UnitPrice          float64       
 8   SalesQuantity      int64         
 9   ReturnQuantity     int64         
 10  ReturnAmount       float64       
 11  DiscountQuantity   int64         
 12  DiscountAmount     float64       
 13  TotalCost          float64       
 14  SalesAmount        float64       
 15  SalesAmountRet     float64       
 16  TotalCostRet       float64       
 17  ProductUnitCost    float64       
 18  DateKey            int64         
 19  GeographyKey       int64         
 20  ContinentName      str           
 

In [15]:
sales_gdp = sales_geo.merge(
    GDP_df,
    left_on=["RegionCountryName","Year"],
    right_on=["CountryKey","Year"],  
    how="left"
)

print(len(sales_geo))
print(len(sales_gdp))
sales_gdp[["RegionCountryName", "CountryKey", "Year"]].head()

2282482
2282482


,RegionCountryName,CountryKey,Year
0,United States,United States,2011
1,United States,United States,2013
2,Turkmenistan,Turkmenistan,2013
3,Turkmenistan,Turkmenistan,2012
4,United States,United States,2013


In [16]:
country_year = (
    sales_gdp
    .groupby(["RegionCountryName", "Year"], as_index=False)
    .agg(
        Revenue=("SalesAmountRet", "sum"),
        TotalCost=("TotalCostRet", "sum"),
        GDP_per_capita=("GDPPerCapita", "mean")
    )
)

country_year["GrossMargin"] = (
    country_year["Revenue"] - country_year["TotalCost"]
)


In [17]:
country_year["Year"] = country_year["Year"].astype(str)
#Year omzetten naar string, anders wordt deze als numerieke waarden geïnterpreteerd door Plotly

fig = px.scatter(
    country_year,
    x="GDP_per_capita",
    #Horziontale as is het GDP per capita
    y="Revenue",
    #Y as is de omzet
    color="Year",
    #Waarden in de scatterplot, de jaren
    hover_name="RegionCountryName",
    #De waarden zijn de landen
    title="Revenue vs GDP per capita",
    size="GrossMargin"
    #Hoe beter de Gross Margin hoe groter de bubbel.
)

fig.update_layout(
    xaxis_title="GDP per capita",
    yaxis_title="Revenue",
    height=450,
    width=1000,
)

fig.update_yaxes(
    tickprefix="€",
    tickformat=",.0f"
)

fig.show()

### Interactie


In [ ]:
#Year slicer
year_options = sorted(sales_df["Date"].dt.year.unique())
year_dropdown = dcc.Dropdown(
id="year-filter",
#Id van het component
options=[
    {"label": str(year), "value": year}
    for year in year_options
],
#Maken van een dictionairy met labels en waarden. De labels worden weergegeven in het dashboard, de waarden zijn te gebruiken in de functie.
placeholder="Select year",
#Tekst die weergegeven wordt als er niks geselecteerd is.
multi=True
#Staat toe dat er meerdere waarden worden geselecteerd.
)
year_options

In [21]:
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H2("Sales Dashboard"),

    year_dropdown,

    dcc.Graph(
        id="revenue-over-time"
    )
])

app.run(debug=True)

In [22]:
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H2("Sales Dashboard"),

    year_dropdown,

    dcc.Graph(
        id="revenue-over-time"
    )
])

@app.callback(
    Output("revenue-over-time", "figure"),
    Input("year-filter", "value")
)
def update_revenue_chart(selected_years):
    df = sales_df.copy()

    if selected_years:
        df = df[df["Date"].dt.year.isin(selected_years)]

    monthly_revenue = (
        df
        .resample("ME", on="Date")
        .agg(Revenue=("SalesAmountRet", "sum"))
        .reset_index()
    )

    monthly_revenue["Year"] = monthly_revenue["Date"].dt.year.astype(str)
    monthly_revenue["Month"] = monthly_revenue["Date"].dt.month

    fig = px.line(
        monthly_revenue,
        x="Month",
        y="Revenue",
        color="Year",
        title="Revenue over time"
    )

    fig.update_layout(
        height=400,
        width=1000
    )
    fig.update_yaxes(
    tickprefix="€",
    tickformat=",.0f"
    )
    #Valuta maken van waarden op de y-as

    fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(1, 13)),
    ticktext=[
        "Jan", "Feb", "Mar", "Apr",
        "May", "Jun", "Jul", "Aug",
        "Sep", "Oct", "Nov", "Dec"
    ]
    )

    return fig

app.run(debug=True)

In [29]:
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H2("Sales Dashboard"),

    year_dropdown,

    dcc.Graph(
        id="revenue-kpi"
    ),

    dcc.Graph(
        id="revenue-over-time"
    )
])
@app.callback(
    Output("revenue-kpi", "figure"),
    Output("revenue-over-time", "figure"),
    Input("year-filter", "value")
)
def update_dashboard(selected_years):
    df = sales_df.copy()

    if selected_years:
        df = df[df["Date"].dt.year.isin(selected_years)]

    revenue = df["SalesAmountRet"].sum()

    kpi_fig = go.Figure(
        go.Indicator(
            mode="number",
            value=revenue,
            title={"text": "Revenue"},
            number={
                "prefix": "€",
                "valueformat": ",.0f"
            }
        )
    )

    kpi_fig.update_layout(
        height=300,
        width=300
    )

    monthly_revenue = (
        df
        .resample("ME", on="Date")
        .agg(Revenue=("SalesAmountRet", "sum"))
        .reset_index()
    )

    monthly_revenue["Year"] = monthly_revenue["Date"].dt.year.astype(str)
    monthly_revenue["Month"] = monthly_revenue["Date"].dt.month

    line_fig = px.line(
        monthly_revenue,
        x="Month",
        y="Revenue",
        color="Year",
        title="Revenue over time"
    )

    line_fig.update_layout(
        height=400,
        width=1000
    )

    line_fig.update_yaxes(
        tickprefix="€",
        tickformat=",.0f"
    )

    line_fig.update_xaxes(
        tickmode="array",
        tickvals=list(range(1, 13)),
        ticktext=[
            "Jan", "Feb", "Mar", "Apr",
            "May", "Jun", "Jul", "Aug",
            "Sep", "Oct", "Nov", "Dec"
        ]
    )

    return kpi_fig, line_fig

app.run(debug=True)

In [ ]:
#Opbouwen van dataframe voor salesdashboard
#Eerst gebruikte dimensies mergen.
sales_dashboard_df = (
    sales_df
    .merge(
        channel_df,
        on="ChannelKey",
        how="left"
    )
    .merge(
        product_df,
        on="ProductKey",
        how="left"
    )
    .merge(
        productcat_df,
        on="ProductSubcategoryKey",
        how="left"
    )
    .merge(
        geography_df,
        on="GeographyKey",
        how="left"
    )
)

sales_dashboard_df["Year"] = sales_dashboard_df["Date"].dt.year

#GDP landen per jaar mergen
sales_dashboard_df = sales_dashboard_df.merge(
    GDP_df,
    left_on=["RegionCountryName", "Year"],
    right_on=["CountryKey", "Year"],
    how="left"
)

In [ ]:
#Slicers opbouwen

#Year slicer
year_options = sorted(sales_dashboard_df["Date"].dt.year.unique())
#Variabele maken met alle unieke jaartallen.

year_dropdown = dcc.Dropdown(
id="year-filter",
#Id van het component
options=[
    {"label": str(year), "value": year}
    for year in year_options
],
#Maken van een dictionairy met labels en waarden. De labels worden weergegeven in het dashboard, de waarden zijn te gebruiken in de functie.
placeholder="Select year",
#Tekst die weergegeven wordt als er niks geselecteerd is.
multi=True
#Staat toe dat er meerdere waarden worden geselecteerd.
)

#Month Slicer
month_names = [
    "Jan", "Feb", "Mar", "Apr",
    "May", "Jun", "Jul", "Aug",
    "Sep", "Oct", "Nov", "Dec"
]

month_options = sorted(sales_dashboard_df["Date"].dt.month.unique())
#Variabele maken met alle unieke jaartallen.

month_dropdown = dcc.Dropdown(
    id="month-filter",
#Id van het component
    options=[
    {"label": month_names[month -1], "value": int(month)}
    for month in month_options
],
#Maken van een dictionairy met labels en waarden. De labels worden weergegeven in het dashboard, de waarden zijn te gebruiken in de functie.
placeholder="Select month(s)",
#Tekst die weergegeven wordt als er niks geselecteerd is.
multi=True
#Staat toe dat er meerdere waarden worden geselecteerd.
)

#Countryname slicer


In [91]:
app = dash.Dash(__name__)

app.layout = html.Div([

    html.H2("Sales Dashboard"),

    html.Div([

        # Linkerkolom: slicers
        html.Div([
            html.H4("Filters"),

            html.Label("Year"),
            year_dropdown,

            html.Label("Month"),
            month_dropdown

        ],
        style={
            "width": "15%",
            "padding": "20px"
        }),

        # Rechterkolom: dashboard
        html.Div([

            # KPI's
            html.Div([
                dcc.Graph(id="revenue-kpi"),
                dcc.Graph(id="cost-kpi"),
                dcc.Graph(id="gross-margin-kpi"),
                dcc.Graph(id="gross-margin-pct-kpi")
            ],
            style={
                "display": "flex"
            }),

            # Eerste rij
            html.Div([
                dcc.Graph(id="revenue-over-time"),
                dcc.Graph(id="revenue-by-channel")
            ],
            style={
                "display": "flex"
            }),

            # Tweede rij
            html.Div([
                dcc.Graph(id="revenue-by-product-category"),
                dcc.Graph(id="gdp-per-capita")
            ],
            style={
                "display": "flex"
            })

        ],
        style={
            "width": "85%"
        })

    ],
    style={
        "display": "flex"
    })
])


@app.callback(
    Output("revenue-kpi", "figure"),
    Output("cost-kpi", "figure"),
    Output("gross-margin-kpi","figure"),
    Output("gross-margin-pct-kpi","figure"),
    Output("revenue-over-time","figure"),
    Output("revenue-by-channel","figure"),
    Output("revenue-by-product-category","figure"),
    Output("gdp-per-capita","figure"),
    Input("year-filter", "value"),
    Input("month-filter", "value")
)
def update_dashboard(selected_years, selected_months):
    #Kopie van dataframe maken om te interpreteren
    df = sales_dashboard_df.copy()

    if selected_years:
        df = df[df["Date"].dt.year.isin(selected_years)]

    if selected_months:
        df = df[df["Date"].dt.month.isin(selected_months)]

    revenue = df["SalesAmountRet"].sum()
    
    total_cost = df["TotalCostRet"].sum()

    grossmargin = revenue-total_cost

    grossmarginpct = (grossmargin/revenue)*100

    #Aggregatie voor revenue by channel
    channel_revenue= (df
    .groupby("ChannelName", as_index=False)
    .agg(Revenue=("SalesAmountRet", "sum"))
    )

    #Aggregatie voor revenue by productcategory
    category_revenue = (
    df
    .groupby("ProductCategory", as_index=False)
    .agg(Revenue=("SalesAmountRet", "sum"))
    .sort_values("Revenue", ascending=True)
    )

    #Aggregatie voor gpd per capita versus revenue
    country_year = (
    df
    .groupby(["RegionCountryName", "Year"], as_index=False)
    .agg(
        Revenue=("SalesAmountRet", "sum"),
        TotalCost=("TotalCostRet", "sum"),
        GDP_per_capita=("GDPPerCapita", "mean")
        )
    )
    country_year["Year"] = country_year["Year"].astype(str)
    country_year["GrossMargin"] = (
    country_year["Revenue"] - country_year["TotalCost"]
    )

    KPIrevenue = go.Figure(
        go.Indicator(
            mode="number",
            value=revenue,
            title={"text": "Revenue"},
            number={
                "prefix": "€",
                "valueformat": ",.0f"
            }
        )
    )

    KPIrevenue.update_layout(
        height=300,
        width=450
    )
    KPIcost = go.Figure(
        go.Indicator(
            mode="number",
            value=total_cost,
            title={"text": "Cost"},
            number={
                "prefix": "€",
                "valueformat": ",.0f"
            }
        )
    )

    KPIcost.update_layout(
        height=300,
        width=450
    )

    KPIGrossmargin = go.Figure(
            go.Indicator(
                mode="number",
                value=grossmargin,
                title={"text": "Gross margin"},
                number={
                    "prefix": "€",
                    "valueformat": ",.0f"
                }
            )
        )
    
    KPIGrossmargin.update_layout(
            height=300,
            width=450
        )
    
    KPIGrossmarginpct = go.Figure(
            go.Indicator(
                mode="number",
                value=grossmarginpct,
                title={"text": "Gross margin %"},
                number={
                    "suffix": "%",
                    "valueformat": ",.2f"
                }
            )
        )
    
    KPIGrossmarginpct.update_layout(
            height=300,
            width=450
        )

    monthly_revenue = (
        df
        .resample("ME", on="Date")
        .agg(Revenue=("SalesAmountRet", "sum"))
        .reset_index()
    )

    monthly_revenue["Year"] = monthly_revenue["Date"].dt.year.astype(str)
    monthly_revenue["Month"] = monthly_revenue["Date"].dt.month

    RevenueOverTimeLine = px.line(
        monthly_revenue,
        x="Month",
        y="Revenue",
        color="Year",
        title="Revenue over time"
    )

    RevenueOverTimeLine.update_layout(
        height=400,
        width=1000
    )

    RevenueOverTimeLine.update_yaxes(
        tickprefix="€",
        tickformat=",.0f"
    )

    RevenueOverTimeLine.update_xaxes(
        tickmode="array",
        tickvals=list(range(1, 13)),
        ticktext=[
            "Jan", "Feb", "Mar", "Apr",
            "May", "Jun", "Jul", "Aug",
            "Sep", "Oct", "Nov", "Dec"
        ]
    )
    RevenueByChannelBar = px.bar(
    #Bar chart maken met behulp van channel_revenue
    channel_revenue,
    x="Revenue",
    y="ChannelName",
    orientation="h",
    title="Revenue by sales channel"
    )

    RevenueByChannelBar.update_layout(
    xaxis_title="Revenue",
    yaxis_title="Sales channel",
    height=400,
    width = 800
    )
    #Titels voor axes instellen.
    #Hoogte en wijdte van de chart instellen.

    RevenueByChannelBar.update_xaxes(
    tickprefix="€",
    tickformat=",.0f"
    )


    RevenueByProductCategory = px.pie(
    category_revenue,
    values="Revenue",
    names="ProductCategory",
    hole=0.8,
    title="Revenue by product category"
    )

    RevenueByProductCategory.update_layout(
    height=450,
    width=800
    )

    
    RevenueByProductCategory.update_traces(
    textposition="outside",
    textinfo="percent+label",
    showlegend=False
    )

    GDPPerCapita = px.scatter(
    country_year,
    x="GDP_per_capita",
    #Horziontale as is het GDP per capita
    y="Revenue",
    #Y as is de omzet
    color="Year",
    #Waarden in de scatterplot, de jaren
    hover_name="RegionCountryName",
    #De waarden zijn de landen
    title="Revenue vs GDP per capita",
    size="GrossMargin"
    #Hoe beter de Gross Margin hoe groter de bubbel.
    )

    GDPPerCapita.update_layout(
        xaxis_title="GDP per capita",
        yaxis_title="Revenue",
        height=450,
        width=1000,
    )

    GDPPerCapita.update_yaxes(
        tickprefix="€",
        tickformat=",.0f"
    )

    

    return KPIrevenue, KPIcost, KPIGrossmargin, KPIGrossmarginpct, RevenueOverTimeLine, RevenueByChannelBar, RevenueByProductCategory, GDPPerCapita


app.run(debug=True)